In [1]:
import glob, json, re
from urllib.parse import urlparse
import numpy as np
import pandas as pd

data = pd.concat(
    [pd.read_csv(f) for f in sorted(glob.glob("output/articles_clean_part*.csv.zip"))],
    ignore_index=True,
)
print("Shape:", data.shape)

CONTRACT = {
    "required_columns": ["url", "title", "content", "source"],
    "optional_columns": ["date_of_publication", "author_name", "description"],
    "min_title_chars_warning": 8,
    "min_content_chars_warning": 200,
    "known_error_patterns": [r"access denied", r"enable javascript", r"captcha",
                             r"robot check", r"403 forbidden", r"page not found"],
}
print(json.dumps(CONTRACT, indent=2))

Shape: (97241, 7)
{
  "required_columns": [
    "url",
    "title",
    "content",
    "source"
  ],
  "optional_columns": [
    "date_of_publication",
    "author_name",
    "description"
  ],
  "min_title_chars_warning": 8,
  "min_content_chars_warning": 200,
  "known_error_patterns": [
    "access denied",
    "enable javascript",
    "captcha",
    "robot check",
    "403 forbidden",
    "page not found"
  ]
}


In [2]:
df = data.copy().reset_index(drop=True)
df["row_index"] = df.index
issues = []

def add_issue(rule_id, severity, row_index, field, message):
    issues.append({"rule_id": rule_id, "severity": severity, "row_index": int(row_index),
                   "field": field, "message": message})

def is_valid_http_url(v):
    if not isinstance(v, str) or not v.strip():
        return False
    try:
        p = urlparse(v.strip())
        return p.scheme.lower() in {"http", "https"} and bool(p.hostname) and "." in p.hostname
    except Exception:
        return False

# schema
for col in CONTRACT["required_columns"]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# required fields (vectorized)
for field in CONTRACT["required_columns"]:
    bad = df[field].isna() | (df[field].astype(str).str.strip() == "")
    for i in df.index[bad]:
        add_issue("FIELD_001", "ERROR", i, field, f"Required field '{field}' is missing or blank.")

# URL
bad_url = ~df["url"].map(is_valid_http_url)
for i in df.index[bad_url]:
    add_issue("URL_001", "ERROR", i, "url", "URL is not a plausible HTTP(S) URL.")

# dates
pub = pd.to_datetime(df["date_of_publication"], errors="coerce", utc=True, format="mixed")
for i in df.index[df["date_of_publication"].notna() & pub.isna()]:
    add_issue("TIME_002", "WARNING", i, "date_of_publication", "Timestamp cannot be parsed.")
for i in df.index[pub > pd.Timestamp("2026-10-29", tz="UTC")]:
    add_issue("TIME_003", "WARNING", i, "date_of_publication", "Publication date is in the future.")

# text
tl = df["title"].fillna("").astype(str).str.len()
cl = df["content"].fillna("").astype(str).str.len()
for i in df.index[(tl > 0) & (tl < CONTRACT["min_title_chars_warning"])]:
    add_issue("TEXT_001", "WARNING", i, "title", "Title is very short.")
for i in df.index[(cl > 0) & (cl < CONTRACT["min_content_chars_warning"])]:
    add_issue("TEXT_002", "WARNING", i, "content", "Content is very short.")
err = re.compile("|".join(CONTRACT["known_error_patterns"]), re.I)
blocked = df["content"].fillna("").astype(str).str.contains(err)
for i in df.index[blocked]:
    add_issue("TEXT_003", "WARNING", i, "content", "Content may be a block/CAPTCHA/error page.")

# duplicates
for i in df.index[df["url"].duplicated(keep=False)]:
    add_issue("DUP_001", "WARNING", i, "url", "URL occurs more than once.")
for i in df.index[df["title"].duplicated(keep=False)]:
    add_issue("DUP_003", "WARNING", i, "title", "Title occurs more than once; inspect.")
for i in df.index[df["content"].duplicated(keep=False)]:
    add_issue("DUP_002", "WARNING", i, "content", "Content occurs more than once.")

issues_df = pd.DataFrame(issues, columns=["rule_id", "severity", "row_index", "field", "message"])
print("Total issues:", len(issues_df))
print(issues_df.groupby(["rule_id", "severity"]).size().to_string() if len(issues_df) else "No issues")

Total issues: 670
rule_id   severity
DUP_003   WARNING     617
TEXT_001  WARNING       1
TEXT_003  WARNING      50
TIME_003  WARNING       2


In [3]:
import os

FATAL_RULES = {"FIELD_001", "URL_001"}
row_rules = issues_df.groupby("row_index")["rule_id"].apply(list).to_dict()

def disposition(i):
    rules = row_rules.get(i, [])
    if any(r in FATAL_RULES for r in rules):
        return "QUARANTINE"
    return "WARNING" if rules else "ACCEPT"

df["validation_issue_codes"] = df["row_index"].map(lambda i: ";".join(row_rules.get(i, [])))
df["validation_disposition"] = df["row_index"].map(disposition)

accepted_df = df[df["validation_disposition"] == "ACCEPT"]
warning_df = df[df["validation_disposition"] == "WARNING"]
quarantine_df = df[df["validation_disposition"] == "QUARANTINE"]
print({"total": len(df), "accepted": len(accepted_df),
       "warning": len(warning_df), "quarantine": len(quarantine_df)})

# --- unit tests ---
def test_valid_url():
    assert is_valid_http_url("https://example.com/story/1")
    assert not is_valid_http_url("not-a-url")
    assert not is_valid_http_url("ftp://example.com/file")
    assert not is_valid_http_url(None)

def test_timestamp():
    assert pd.notna(pd.to_datetime("2026-10-27T09:30:00Z", errors="coerce", utc=True))
    assert pd.isna(pd.to_datetime("yesterday maybe", errors="coerce", utc=True))

def test_no_fatal_rows():
    assert len(quarantine_df) == 0

test_valid_url(); test_timestamp(); test_no_fatal_rows()
print("All tests passed.")

# --- summary + exports ---
summary = {
    "input_rows": int(len(df)),
    "total_issues": int(len(issues_df)),
    "issues_by_rule": {k: int(v) for k, v in issues_df.groupby("rule_id").size().items()},
    "dispositions": {"accept": len(accepted_df), "warning": len(warning_df),
                     "quarantine": len(quarantine_df)},
}
os.makedirs("output/validation", exist_ok=True)
issues_df.to_csv("output/validation/news_validation_issues.csv", index=False, encoding="utf-8-sig")
warning_df[["row_index", "source", "url", "title", "validation_issue_codes"]].to_csv(
    "output/validation/news_warning_review.csv", index=False, encoding="utf-8-sig")
quarantine_df[["row_index", "source", "url", "title", "validation_issue_codes"]].to_csv(
    "output/validation/news_quarantine.csv", index=False, encoding="utf-8-sig")
with open("output/validation/news_validation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
for p in sorted(os.listdir("output/validation")):
    print(p, round(os.path.getsize(f"output/validation/{p}") / 1024**2, 2), "MB")

{'total': 97241, 'accepted': 96571, 'warning': 670, 'quarantine': 0}
All tests passed.
{
  "input_rows": 97241,
  "total_issues": 670,
  "issues_by_rule": {
    "DUP_003": 617,
    "TEXT_001": 1,
    "TEXT_003": 50,
    "TIME_003": 2
  },
  "dispositions": {
    "accept": 96571,
    "warning": 670,
    "quarantine": 0
  }
}
news_quarantine.csv 0.0 MB
news_validation_issues.csv 0.04 MB
news_validation_summary.json 0.0 MB
news_warning_review.csv 0.11 MB
